# Part 4: Structured Follow-Up Note Analysis

Using Gemini Flash with JSON mode and Pydantic validation to turn the first 10 call-centre notes into structured root causes, then checking prompt sensitivity and cross-tabulating against reminders.

In [ ]:
!pip install -q google-genai pydantic

In [ ]:
import json
import time
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

## Task 4.1 — Single validated API call

Enter the Gemini API key at runtime (never save it in the notebook, and clear cell output before downloading).

In [ ]:
API_KEY = input("Paste your Gemini API key: ")
client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-2.5-flash"  # Update if your Session 22.1 notebook uses a different Flash model
api_calls = 0

ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

SYSTEM_TEXT = (
    "You classify outpatient clinic follow-up call notes accurately and concisely. "
    "Always return valid JSON matching the requested fields."
)

def build_craft_prompt(note_text):
    return (
        "Context: CareLoop Clinics' call-centre staff wrote a brief note after phoning a patient who missed an appointment.\n"
        "Role: Clinic operations analyst.\n"
        "Action: Classify the note into exactly one root_cause from this allowed list: "
        f"{', '.join(ALLOWED_CAUSES)}. "
        "If the note is vague, ambiguous, or does not clearly fit the first five categories, choose 'other' rather than guessing. "
        "Provide a confidence score between 0 and 1, a summary of at most 25 words, and one practical recommended_action.\n"
        "Tone: Neutral and operational.\n"
        "Format: JSON with keys root_cause, confidence, summary, recommended_action.\n\n"
        f"Note: {note_text}"
    )

def analyse_note(note_text, prompt_fn=build_craft_prompt):
    global api_calls
    prompt = prompt_fn(note_text)
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_TEXT,
        temperature=0,
        max_output_tokens=1000,
        response_mime_type="application/json",
        response_schema=SCHEMA,
        thinking_config=types.ThinkingConfig(thinking_level="minimal"),
    )
    api_calls += 1
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=config,
    )
    raw_text = response.text
    data = json.loads(raw_text)
    return data, raw_text

notes_df = pd.read_csv("data/followup_notes.csv")
n001_text = notes_df.loc[notes_df["note_id"] == "N001", "note_text"].iloc[0]
n001_data, _ = analyse_note(n001_text)
print("N001 parsed output:", n001_data)

## Task 4.2 — Validate and process the first 10 notes

In [ ]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

results = []
first_10 = notes_df.head(10)

for idx, row in first_10.iterrows():
    note_id = row["note_id"]
    appt_id = row["appointment_id"]
    note_text = row["note_text"]

    if pd.isna(note_text) or str(note_text).strip() == "":
        results.append({
            "note_id": note_id,
            "appointment_id": appt_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": "",
        })
        continue

    raw_text = ""
    data = {}
    status = "invalid"
    try:
        data, raw_text = analyse_note(note_text)
        insight = FollowupInsight(**data)
        if insight.root_cause in ALLOWED_CAUSES:
            status = "valid"
    except Exception as e:
        print(f"{note_id} validation/API issue: {e}")

    results.append({
        "note_id": note_id,
        "appointment_id": appt_id,
        "root_cause": data.get("root_cause", ""),
        "confidence": data.get("confidence", ""),
        "summary": data.get("summary", ""),
        "recommended_action": data.get("recommended_action", ""),
        "validation_status": status,
        "raw_response": raw_text,
    })

    time.sleep(13)

analysed_df = pd.DataFrame(results)
analysed_df.to_csv("notes_analysed.csv", index=False)
analysed_df[["note_id", "appointment_id", "root_cause", "confidence", "validation_status"]]

## Task 4.3 — Break it on purpose and compare prompts

In [ ]:
# (a) Deliberately bad payloads
bad_1 = {"root_cause": "other", "confidence": 1.7, "summary": "Patient forgot.", "recommended_action": "Send SMS"}
bad_2 = {"root_cause": "other", "confidence": 0.8, "summary": "Patient forgot."}

for label, payload in [("confidence=1.7", bad_1), ("missing recommended_action", bad_2)]:
    try:
        FollowupInsight(**payload)
    except ValidationError as e:
        print(f"--- {label} ---")
        print(e)

# (b) Compare V1 vs V2 on N007 (Hinglish) and N009 (ambiguous)
def build_v1_prompt(note_text):
    return f"Classify this note\n\nNote: {note_text}"

cmp_rows = []
for nid in ["N007", "N009"]:
    txt = notes_df.loc[notes_df["note_id"] == nid, "note_text"].iloc[0]
    d1, _ = analyse_note(txt, prompt_fn=build_v1_prompt)
    time.sleep(13)
    # Reuse V2 result from Task 4.2 to save API calls
    v2_row = analysed_df.loc[analysed_df["note_id"] == nid].iloc[0]
    cmp_rows.append({"note_id": nid, "prompt": "V1", "root_cause": d1.get("root_cause"), "allowed": d1.get("root_cause") in ALLOWED_CAUSES})
    cmp_rows.append({"note_id": nid, "prompt": "V2 (CRAFT)", "root_cause": v2_row["root_cause"], "allowed": v2_row["root_cause"] in ALLOWED_CAUSES})

cmp_table = pd.DataFrame(cmp_rows)
print("\nPrompt comparison:")
print(cmp_table.to_string(index=False))

# (c) Total API calls used
print(f"\nTotal API calls used across Tasks 4.1-4.3: {api_calls}")

I would deploy **V2 (CRAFT)** because it explicitly constrains the model to the six allowed labels and tells it to fall back to `other` on ambiguous notes instead of inventing a new category.

## Task 4.4 — Cross-tabulate root causes against reminders

In [ ]:
clean_df = pd.read_csv("appointments_clean.csv")
merged = analysed_df.merge(clean_df, on="appointment_id", how="left")
print("Merged rows:", len(merged))

ct = pd.crosstab(merged["root_cause"], merged["reminder_sent"], margins=True)
print("\nroot_cause vs reminder_sent:")
ct